# Lost or Inaccessible? — MASTER Colab experiment notebook

**Upload the supplied project ZIP. Run the complete five-seed CIFAR-10 study. Download high-resolution PNG figures.**

1. In Colab choose **Runtime → Change runtime type → a GPU**, if available. CPU fallback is supported.
2. Choose **Runtime → Run all**. Authorize your own Google Drive account when prompted.
3. The upload cell asks for `lost-or-inaccessible.zip` on the first run. Upload the **updated ZIP supplied with this notebook**.
4. Leave the full experiment cell running. Re-run this same notebook after a runtime interruption to resume from Drive artifacts.
5. The last cells verify all stages, render **600-DPI PNGs only**, display a gallery, and create downloadable ZIPs.

The full study includes all five seeds, the full neuron/edge budgets, repeated random controls, linear and nonlinear probes, restoration, and statistics. It can span multiple Colab sessions. This notebook contains **no prefilled research results**. Drive storage capacity does not guarantee GPU access or runtime duration.

Google documentation: [Colab usage and runtime limits](https://research.google.com/colaboratory/faq.html). No paid API, remote agent, keep-alive script, or extra dataset is used.

## 1 · Environment and free GPU/CPU detection
Keep Colab’s preinstalled matched PyTorch/torchvision packages. Install the remaining project dependencies from PyPI.

In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
os.environ.setdefault('MPLCONFIGDIR', '/content/loi_matplotlib_cache')
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
               'numpy>=1.23', 'scipy>=1.10', 'scikit-learn>=1.2', 'pandas>=1.5',
               'matplotlib>=3.8', 'PyYAML>=6', 'pytest>=7', 'Pillow>=9'], check=True)
if importlib.util.find_spec('torch') is None or importlib.util.find_spec('torchvision') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision'], check=True)
try:
    import torch, torchvision
except Exception as exc:
    raise RuntimeError('PyTorch/torchvision did not import. Use a fresh standard Colab runtime with its matched packages, then rerun.') from exc
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU fallback selected. The complete study will be much slower.')


## 2 · Mount Drive and set persistent storage
Data, uploaded project archives, configuration, checkpoints and outputs are kept in your own Drive. Authorize the mount yourself. On a later runtime, use the same account and storage path.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PROJECT = Path('/content/drive/MyDrive/lost-or-inaccessible')
DATA_DIR = DRIVE_PROJECT / 'datasets'
OUTPUT_DIR = DRIVE_PROJECT / 'results'
UPLOAD_DIR = DRIVE_PROJECT / 'uploads'
EXPORT_DIR = DRIVE_PROJECT / 'exports'
for path in [DATA_DIR, OUTPUT_DIR, UPLOAD_DIR, EXPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)
print('Persistent project folder:', DRIVE_PROJECT)


## 3 · Upload/reuse the project ZIP

On the first run, select the supplied **updated** `lost-or-inaccessible.zip`. The archive is copied to Drive and its SHA-256 recorded. On resumed sessions the same ZIP is reused automatically. Set `UPLOAD_NEW_ZIP = True` only when you deliberately want a different source version; source changes create a different experiment directory.

The source is extracted into the temporary Colab runtime. Existing dataset/checkpoint folders in an uploaded ZIP are ignored; the verified Drive dataset and run outputs are managed separately.

In [ ]:
import hashlib, json, shutil, stat, zipfile
from pathlib import PurePosixPath
from google.colab import files
UPLOAD_NEW_ZIP = False
ZIP_PATH_OVERRIDE = ''  # Optional existing .zip path in your mounted Drive.
last_zip = UPLOAD_DIR / 'LAST_PROJECT_ZIP.txt'
if ZIP_PATH_OVERRIDE:
    project_zip = Path(ZIP_PATH_OVERRIDE)
elif last_zip.exists() and not UPLOAD_NEW_ZIP:
    project_zip = Path(last_zip.read_text().strip())
else:
    print('Select the updated lost-or-inaccessible.zip supplied with this master notebook.')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one project ZIP. Rerun this cell to try again.')
    name, payload = next(iter(uploaded.items()))
    if not name.lower().endswith('.zip'):
        raise ValueError('The project must be a .zip file.')
    digest = hashlib.sha256(payload).hexdigest()
    project_zip = UPLOAD_DIR / f'project_{digest[:16]}.zip'
    if not project_zip.exists():
        temporary = project_zip.with_suffix('.zip.partial')
        temporary.write_bytes(payload)
        temporary.replace(project_zip)
    last_zip.write_text(str(project_zip))
    del payload, uploaded
if not project_zip.is_file():
    raise FileNotFoundError(f'ZIP not found: {project_zip}. Set UPLOAD_NEW_ZIP=True to upload again.')
with project_zip.open('rb') as stream:
    hasher = hashlib.sha256()
    for block in iter(lambda: stream.read(1024*1024), b''):
        hasher.update(block)
    digest = hasher.hexdigest()
WORK_ROOT = Path('/content/loi_sources') / digest[:16]
WORK_ROOT.mkdir(parents=True, exist_ok=True)
extraction_marker = WORK_ROOT / '.extraction_complete'
if not extraction_marker.exists():
    skip = {'__MACOSX', '__pycache__', '.git', '.venv', '.pytest_cache', 'datasets', 'results'}
    with zipfile.ZipFile(project_zip) as archive:
        members = []
        for item in archive.infolist():
            rel = PurePosixPath(item.filename)
            if rel.is_absolute() or '..' in rel.parts or '\\\\' in item.filename:
                raise ValueError(f'Unsafe ZIP member: {item.filename}')
            if stat.S_ISLNK(item.external_attr >> 16):
                raise ValueError('Symlinks are not accepted in the project ZIP.')
            if skip.intersection(rel.parts) or item.is_dir():
                continue
            members.append(item)
        size = sum(item.file_size for item in members)
        if size > shutil.disk_usage(WORK_ROOT).free:
            raise RuntimeError('Insufficient Colab runtime disk space to extract the source ZIP.')
        for item in members:
            destination = WORK_ROOT.joinpath(*PurePosixPath(item.filename).parts)
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(item) as source, destination.open('wb') as target:
                shutil.copyfileobj(source, target)
    extraction_marker.write_text(digest)
candidates = [p.parent for p in WORK_ROOT.rglob('run.py')
              if (p.parent/'configs/colab.yaml').exists() and (p.parent/'models/mlp.py').exists()]
if len(candidates) != 1:
    raise RuntimeError(f'Expected exactly one research project in the ZIP; found {len(candidates)}.')
PROJECT_DIR = candidates[0]
if '--report-only' not in (PROJECT_DIR/'run.py').read_text() or 'PNG-only' not in (PROJECT_DIR/'analysis/plots.py').read_text():
    raise RuntimeError('This ZIP is an older version. Set UPLOAD_NEW_ZIP=True and upload the updated supplied project ZIP.')
print('Source ZIP:', project_zip)
print('SHA-256:', digest)
print('Project directory:', PROJECT_DIR)


## 4 · Preflight software tests
These tests use synthetic fixtures strictly to verify the software. They are not research observations. Failure stops the notebook before the main study. Fixture plots use low DPI to keep this check quick; research figures are rendered at 600 DPI below.

In [ ]:
test_environment = dict(os.environ, LOI_FIGURE_DPI='100')
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=PROJECT_DIR,
               env=test_environment, check=True)
print('Software preflight passed. Research experiments have not started yet.')


## 5 · Download/reuse and verify official CIFAR-10
If you already ran the Drive-download notebook, this reuses `MyDrive/lost-or-inaccessible/datasets`. The complete original files remain unchanged. Only Cats (3→0) and Dogs (5→1) enter the research pipeline. A verification record is written only after the official file checks pass.

In [ ]:
from collections import Counter
from datetime import datetime, timezone
from torchvision.datasets import CIFAR10
train = CIFAR10(str(DATA_DIR), train=True, download=True)
test = CIFAR10(str(DATA_DIR), train=False, download=True)
assert train._check_integrity() and test._check_integrity()
train_counts, test_counts = Counter(train.targets), Counter(test.targets)
assert train_counts[3] == train_counts[5] == 5000
assert test_counts[3] == test_counts[5] == 1000
verification = dict(status='verified', verified_at_utc=datetime.now(timezone.utc).isoformat(),
                    source=CIFAR10.url, cat_dog_train=10000, cat_dog_test=2000,
                    torchvision_file_checksums='passed', directory=str(DATA_DIR))
(DATA_DIR/'DATASET_VERIFIED.json').write_text(json.dumps(verification, indent=2))
print(json.dumps(verification, indent=2))
del train, test


## 6 · Confirm the complete study configuration
These are the **full study** settings. This notebook does not silently switch to a pilot. Both linear and nonlinear probes are fitted independently for every intervention condition. The printed condition/fit counts help explain why this may require several sessions.

The original models select checkpoints by validation loss. Ranking uses analysis examples; the official test subset is reserved for reported outcomes. Read `docs/METHODOLOGY.md` before interpreting information-loss claims.

In [ ]:
import yaml, math
base = yaml.safe_load((PROJECT_DIR/'configs/base.yaml').read_text())
base.update(yaml.safe_load((PROJECT_DIR/'configs/colab.yaml').read_text()))
assert base['seeds'] == [42, 123, 456, 789, 2026]
assert base['experiments'] == [f'e{i}' for i in range(1, 11)]
assert base['probe_families'] == ['linear', 'nonlinear'] and base['probe_conditions'] == 'all'
print(yaml.safe_dump(base, sort_keys=False))
repetitions = base['random_repeats']
neuron_conditions = 3 * 2 * len(base['neuron_budgets']) * (3 + 2*repetitions)
edge_conditions = 5 * 2 * len(base['edge_budgets']) * (2 + 2*repetitions)
print('Per training seed:', neuron_conditions, 'neuron conditions and', edge_conditions, 'edge conditions')
print('Total intervention conditions:', len(base['seeds'])*(neuron_conditions+edge_conditions))
# Probed-layer multiplicities: neurons 3+2+1; edges global 3, hidden 3+2+1, readout reference 1.
neuron_fits = 2*len(base['neuron_budgets'])*(3+2*repetitions)*6
edge_fits = 2*len(base['edge_budgets'])*(2+2*repetitions)*10
probe_fits = len(base['seeds'])*len(base['probe_families'])*(3+neuron_fits+edge_fits)
print('Configured probe fits (including explicitly labeled readout references):', probe_fits)
print('Full execution is resumable. No completion time or GPU availability is guaranteed.')


## 7 · RUN THE WHOLE EXPERIMENT — E1–E9
This cell executes the complete configured study in dependency order, with live console output and a persistent log. E10 reporting runs in the next cell so that figures are rendered once at full resolution.

Keep the notebook open and interact with it normally. If Colab ends the runtime, reconnect and rerun the same notebook/source ZIP; existing completed stages, condition metrics, probe fits and training checkpoints are reused. Do not edit Python source or configuration midway through a confirmatory run. The full run includes tens of thousands of probe fits, so a single free Colab session may not finish it.

You may explicitly set `EXPERIMENT = 'e1'` etc. for staged execution, but the default is `'all'`. A staged run is not reported as a completed whole study.

In [ ]:
EXPERIMENT = 'all'  # 'all' runs the full configured study. Change only for deliberate staged execution.
FIGURE_DPI = 600
command = [sys.executable, '-u', 'run.py', '--profile', 'colab',
           '--experiment', EXPERIMENT, '--device', DEVICE,
           '--data-root', str(DATA_DIR), '--output', str(OUTPUT_DIR),
           '--download', '--skip-report', '--figure-dpi', str(FIGURE_DPI)]
log_file = DRIVE_PROJECT / 'colab_pipeline_latest.log'
print('Command:', ' '.join(command))
print('Persistent console log:', log_file)
RUN = None
process = subprocess.Popen(command, cwd=PROJECT_DIR, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, bufsize=1, env=os.environ.copy())
try:
    with log_file.open('a', buffering=1) as log:
        log.write('\n=== SESSION START ' + datetime.now(timezone.utc).isoformat() + ' ===\n')
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            if line.startswith('Run directory: '):
                run_path = line.split('Run directory: ', 1)[1].split(' | device:', 1)[0].strip()
                RUN = Path(run_path)
                (DRIVE_PROJECT/'LAST_RUN.txt').write_text(str(RUN))
        return_code = process.wait()
except KeyboardInterrupt:
    process.terminate()
    try:
        process.wait(timeout=20)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()
    raise RuntimeError('Run interrupted. Completed artifacts remain on Drive; rerun the identical configuration to resume.')
if return_code:
    raise RuntimeError(f'Pipeline stopped with exit code {return_code}. Inspect {log_file} and the *.failed.json files; do not interpret partial outputs as a completed study.')
if RUN is None:
    raise RuntimeError('The pipeline did not report a run directory. Inspect the persistent console log.')
print('\nRequested experiment execution returned successfully. Next: verify completion and generate PNG figures.')


## 8 · Verify completion and render the PNG report — E10
This can also be rerun after an interruption to inspect a partial run. It reads existing measurements and does not train models. Missing experiments remain marked incomplete; missing figure families are never fabricated.

In [ ]:
if 'RUN' not in globals() or RUN is None:
    pointer = DRIVE_PROJECT/'LAST_RUN.txt'
    if not pointer.exists():
        raise RuntimeError('No run was recorded. Execute the experiment cell first.')
    RUN = Path(pointer.read_text().strip())
if not (RUN/'config.json').exists():
    raise RuntimeError(f'Not a valid run directory: {RUN}')
run_config = json.loads((RUN/'config.json').read_text())
import pandas as pd
records = []
for seed in run_config['seeds']:
    for stage in [f'e{i}' for i in range(1, 9)]:
        marker = RUN/f'seed_{seed}'/f'{stage}.done.json'
        records.append(dict(seed=seed, stage=stage.upper(), status='COMPLETE' if marker.exists() else 'NOT COMPLETE'))
completion = pd.DataFrame(records)
display(completion.pivot(index='seed', columns='stage', values='status'))
subprocess.run([sys.executable, '-u', 'run.py', '--report-only', str(RUN),
                '--figure-dpi', str(FIGURE_DPI)], cwd=PROJECT_DIR, check=True)
full_complete = bool((completion.status=='COMPLETE').all() and (RUN/'e9.done.json').exists())
print('E9 statistics:', 'COMPLETE' if (RUN/'e9.done.json').exists() else 'NOT COMPLETE')
print('WHOLE STUDY:', 'COMPLETE' if full_complete else 'INCOMPLETE — do not present partial results as the full study')
figure_status = json.loads((RUN/'figures/figure_status.json').read_text())
print('Figure families:', json.dumps(figure_status['families'], indent=2))
print('PNG directory:', RUN/'figures')


## 9 · Inspect the figures and tables
All experiment figures are PNG, 600 DPI. The gallery uses scaled previews; downloaded PNGs retain full resolution. Panels distinguish target class, layer and probe family. Legends are outside data panels and captions explain aggregation and uncertainty.

A layout boundary check is saved in `figure_manifest.json`; it detects text outside the allocated canvas, but cannot certify every visual overlap. Inspect the PNGs before manuscript use. Figure families are omitted if their measured inputs or independent training-seed replications are missing.

In [ ]:
from IPython.display import display, Image, Markdown
from PIL import Image as PILImage
import io
png_files = sorted((RUN/'figures').glob('*.png'))
if not png_files:
    raise RuntimeError('No PNG figures found; inspect the reporting cell output.')
manifest = json.loads((RUN/'figures/figure_manifest.json').read_text())
figure_rows = []
for path in png_files:
    with PILImage.open(path) as im:
        width, height = im.size
    record = manifest.get(path.name, {})
    figure_rows.append(dict(file=path.name, width_px=width, height_px=height,
                            dpi=record.get('dpi'), layout_flags=len(record.get('text_outside_layout', []))))
display(pd.DataFrame(figure_rows))
for path in png_files:
    display(Markdown('### ' + path.stem.replace('_', ' ')))
    with PILImage.open(path) as original:
        preview = original.copy()
        preview.thumbnail((1600, 1100))
        buffer = io.BytesIO()
        preview.save(buffer, format='PNG')
    display(Image(data=buffer.getvalue(), width=min(1100, preview.width)))
for table in ['T2', 'T4', 'T6', 'T8', 'T9', 'T10']:
    path = RUN/'tables'/f'{table}.csv'
    if path.exists() and path.stat().st_size > 2:
        display(Markdown('### ' + table + ' — first 15 measured rows'))
        display(pd.read_csv(path).head(15))


## 10 · Create download bundles
`FIGURES_PNG_600DPI.zip` contains **only PNG image files**. The separate report bundle contains the figure manifest, measured CSV tables, run configuration, completion records and relevant statistics. Original checkpoints/probes remain on Drive and are not forced into a huge browser download.

The bundles are always saved to Drive. Browser downloads are requested by default; set `DOWNLOAD_TO_COMPUTER = False` to retain only the Drive copies.

In [ ]:
DOWNLOAD_TO_COMPUTER = True
figures_zip = EXPORT_DIR / f'{RUN.name}_FIGURES_PNG_{FIGURE_DPI}DPI.zip'
report_zip = EXPORT_DIR / f'{RUN.name}_MEASURED_REPORT.zip'
with zipfile.ZipFile(figures_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in png_files:
        archive.write(path, path.name)
with zipfile.ZipFile(report_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted((RUN/'tables').glob('*')):
        if path.is_file():
            archive.write(path, str(path.relative_to(RUN)))
    for path in sorted((RUN/'figures').glob('*.json')):
        archive.write(path, str(path.relative_to(RUN)))
    for name in ['config.json','environment.json','splits.json','execution_status.json',
                 'e9.done.json','e10.done.json','robustness.csv','paired_tests.csv',
                 'paired_image_bootstrap.csv','circuit_stability.csv','performance_matched_probes.csv']:
        path=RUN/name
        if path.exists():
            archive.write(path, name)
    for seed in run_config['seeds']:
        for path in sorted((RUN/f'seed_{seed}').glob('e*.csv')):
            archive.write(path, str(path.relative_to(RUN)))
export_status = dict(run=str(RUN), whole_study_complete=full_complete, figure_format='PNG',
                     figure_dpi=FIGURE_DPI, number_of_png_figures=len(png_files),
                     figures_zip=str(figures_zip), measured_report_zip=str(report_zip))
(EXPORT_DIR/f'{RUN.name}_EXPORT_STATUS.json').write_text(json.dumps(export_status, indent=2))
print(json.dumps(export_status, indent=2))
if DOWNLOAD_TO_COMPUTER:
    files.download(str(figures_zip))
    files.download(str(report_zip))
print('Finished. Inspect completion status before writing any scientific conclusions.')


## Resume / troubleshoot

- **Colab runtime ended:** reconnect, choose the same Drive account, leave `UPLOAD_NEW_ZIP=False`, then rerun. Source/configuration fingerprints prevent accidentally mixing changed runs.
- **A stage failed:** inspect `colab_pipeline_latest.log` and the stage’s `.failed.json`. Correct the actual failure before rerunning. There is no synthetic-data fallback.
- **Only want to regenerate figures:** run setup/source cells, then reporting/gallery/export cells. They use `LAST_RUN.txt`; do not rerun training just to change PNG resolution.
- **Drive quota or I/O issues:** all outputs are under one dedicated project folder. Several large runs and thousands of probe artifacts can be substantial. The PNG ZIP excludes model/probe checkpoints.
- **A figure family is missing:** inspect `figure_status.json` and completion markers. F12 needs at least two independently trained models. The primary protocol still requires all five configured seeds.
- **Data already in Drive:** automatic checksum validation reuses it. Do not upload a different dataset in place of CIFAR-10.
- **Interpreting probes:** a successful post-intervention probe demonstrates decodability by that decoder family. Failure does not prove all information was erased. Output-edge-only ablations are labeled unaffected readout references and excluded from post-intervention recovery figures.

This notebook was prepared and locally syntax-/software-tested; it has not been executed in your authenticated Colab account. Your real experiment measurements will appear only after you run it.